# Rung 0 — The growth rate from a straight line

**Question.** How fast is this epidemic growing?

The cheapest honest answer. While susceptibles are still essentially the whole
population, incidence grows like `exp(r t)`, so `log(cases)` is a straight line and
a straight line has a closed-form fit — no solver, no starting values, no convergence.

What it buys is one number, and it is worth being exact about which one: `r = beta - gamma`,
a *combination* of the two rates. Every `(beta, gamma)` pair with the same difference fits
the early data identically. That is not a limitation of the method; it is a property of the
data, and rung 3 makes it explicit.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from inferladder import default_sir_dataset
from inferladder.inference import exponential_growth_rate

dataset = default_sir_dataset()
truth = dataset.truth
truth

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, scale in zip(axes, ("linear", "log"), strict=True):
    ax.plot(dataset.t, dataset.y, ".", ms=4, color="#33628f")
    ax.set(xlabel="day", ylabel="reported cases", yscale=scale, title=f"{scale} scale")
    ax.grid(alpha=0.3)
axes[1].set_ylim(bottom=0.5)
fig.suptitle("The same data twice. Only the log scale shows a straight early phase.")
fig.tight_layout()

## The fit

The window is a modelling choice, not a detail. Too early and the counts are tiny and
noise-dominated; too late and susceptible depletion has already bent the curve, which
biases `r` downward.

In [ ]:
window = (5.0, 25.0)
fit = exponential_growth_rate(dataset.t, dataset.y, window=window)
low, high = fit.confidence_interval()

print(f"growth rate r = {fit.slope:.4f}   95% CI [{low:.4f}, {high:.4f}]")
print(f"true beta - gamma = {truth['beta'] - truth['gamma']:.4f}")
print(f"doubling time = {fit.doubling_time():.1f} days       R^2 = {fit.r_squared:.3f}")

In [ ]:
inside = (dataset.t >= window[0]) & (dataset.t <= window[1]) & (dataset.y > 0)
line_t = np.linspace(*window, 50)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(dataset.t, dataset.y, ".", ms=4, color="#c3ccd6", label="outside window")
ax.plot(dataset.t[inside], dataset.y[inside], ".", ms=7, color="#33628f", label="fitted")
ax.plot(
    line_t,
    np.exp(fit.intercept + fit.slope * line_t),
    "-",
    lw=2,
    color="#c1502e",
    label=f"exp fit, r = {fit.slope:.3f}",
)
ax.set(xlabel="day", ylabel="reported cases", yscale="log", xlim=(0, 60), ylim=(0.5, None))
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

## The window is doing work

Sweeping the end of the window shows the bias arriving. This is the plot that stops
rung 0 from being oversold.

In [ ]:
ends = np.arange(15.0, 61.0, 2.0)
rates, errs = [], []
for end in ends:
    f = exponential_growth_rate(dataset.t, dataset.y, window=(5.0, end))
    rates.append(f.slope)
    errs.append(f.slope_stderr)
rates, errs = np.array(rates), np.array(errs)

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.fill_between(ends, rates - 1.96 * errs, rates + 1.96 * errs, alpha=0.25, color="#33628f")
ax.plot(ends, rates, "-o", ms=3, color="#33628f", label="estimated r")
ax.axhline(truth["beta"] - truth["gamma"], ls="--", color="#c1502e", label="true beta - gamma")
ax.set(xlabel="last day included in the window", ylabel="estimated growth rate")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

The estimate is unbiased only while the exponential approximation holds. Extend the
window and `r` decays toward zero — not because growth slowed in the model, but because
the straight line is being fitted to something that has stopped being straight.

## When to stop here

**This rung is enough if the decision is about timing.** A doubling time answers "how long
until we run out of beds at the current rate", and it does so with one regression and no
mechanistic commitment.

**It is not enough to ask anything counterfactual.** `beta` and `gamma` never appear
separately, so "what if transmission dropped 30%" has no expression here. Answering that
needs the whole curve — rung 1.